# Evaluate thermal only capabilities of PyCheMelt

This notebook showcases the capabilities of PyCheMelt to handle DSF data containing multiple concentrations of protein for analysis. It can handle monomers or oligomers up to tetramers and two state or three state models of defolding.

In [1]:
from pychemelt import ThermalOligomer

import numpy as np
import pandas as pd

from pychemelt.utils.math import linear_baseline, exponential_baseline

from pychemelt.utils.plotting import plot_unfolding, plot_baselines

from pychemelt.utils.signals import (
    map_two_state_model_to_signal_fx,
    map_three_state_model_to_signal_fx,
)

from pychemelt.utils.processing import (
    get_colors_from_numeric_values,
    combine_sequences
)

from pychemelt.utils.plotting import *

from scripts import display_figure_static

## Two State

In [2]:
def aux_create_pychem_sim_two_state(params, concs, temp_range, model, n_residues, rng_seed=2):

    """
    Generate a Pychemelt ThermalOligomer object with simulated data with a two state unfolding model.

    Parameters
    ---------
    params : dict
        dictionary containing the parameters for the simulated signal (e.g., Tm, DH)
    concs : list
        List of concentrations to simulate
    temp_range : list
        Range of temperatures to simulate
    model : str
        Which protein model to use (choices rang from Monomer to Tetramer)
    n_residues : int
        How many residues the simulated protein has

    Returns
    ------
        pychem.ThermalOligomer()
    """

    rng = np.random.default_rng(rng_seed)

    temp_range_K = temp_range + 273.15


    signal_list = []
    temp_list   = []

    signal_fx = map_two_state_model_to_signal_fx(model)

    for i,C in enumerate(concs):

        y = signal_fx(temp_range_K, C, **params)

        # Add gaussian error to simulated signal
        y += rng.normal(0, 0.002*1e-3, len(y))

        # Add error to the initial signal to model variance across positions
        #y *= rng.uniform(0.9,1.1)

        signal_list.append(y)
        temp_list.append(temp_range)

    pychem_sim = ThermalOligomer()

    pychem_sim.signal_dic['Simulated Signal'] = signal_list
    pychem_sim.temp_dic['Simulated Signal']   = [temp_range for _ in range(len(concs))]

    pychem_sim.set_model(model)

    pychem_sim.conditions = concs

    pychem_sim.global_min_temp = np.min(temp_range)
    pychem_sim.global_max_temp = np.max(temp_range)

    pychem_sim.set_concentrations()

    pychem_sim.set_signal('Simulated Signal')

    pychem_sim.select_conditions()
    pychem_sim.expand_multiple_signal()

    pychem_sim.estimate_baseline_parameters(
        native_baseline_type='linear',
        unfolded_baseline_type='exponential',
        window_range_native=12,
        window_range_unfolded=12
    )


    pychem_sim.n_residues = n_residues # only for cp initial guess
    pychem_sim.guess_Cp()

    return pychem_sim

# Centralized test constants
RNG_SEED = 2
TEMP_START = 20.0
TEMP_STOP = 90.0
N_TEMPS = 150
CONCS = np.arange(10, 60, 10)*1e-6
MAX_POINTS = 400


# Model / ground-truth parameters
DHm_VAL = 250
Tm_VAL = 70
CP0_VAL = 1.0


INTERCEPT_I = 15

INTERCEPT_N = 24
SLOPE_N = -0.27
C_N_VAL = 0
INTERCEPT_U = -4
SLOPE_U = 80.5
EXPONENT_U = 0.0224
C_U_VAL = 0

params = {
    'dHm': DHm_VAL,
    'Tm': Tm_VAL+273.15,
    'Cp': CP0_VAL,
    'p1_N': C_N_VAL,
    'p2_N': INTERCEPT_N,
    'p3_N': SLOPE_N,
    'p4_N': 0,
    'p1_U': C_U_VAL,
    'p2_U': INTERCEPT_U,
    'p3_U': SLOPE_U,
    'p4_U': EXPONENT_U,
    'baseline_N_fx':linear_baseline,
    'baseline_U_fx':exponential_baseline,
}

concs = CONCS
temp_range  = np.linspace(TEMP_START, TEMP_STOP, N_TEMPS)

def parameter_fitting_evaluation_two_state(pychem_sim):

    if pychem_sim.global_global_global_fit_done == True:
        actual_params = [Tm_VAL, DHm_VAL, CP0_VAL, INTERCEPT_N, INTERCEPT_U, SLOPE_N, SLOPE_U, EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]),'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2)})

    if pychem_sim.global_global_fit_done == True:
        actual_params = [Tm_VAL, DHm_VAL, CP0_VAL] + len(pychem_sim.conditions) * [INTERCEPT_N] + len(pychem_sim.conditions) * [INTERCEPT_U] + [SLOPE_N, SLOPE_U, EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]), 'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2)})


    if pychem_sim.global_fit_done == True:
        actual_params = [Tm_VAL, DHm_VAL, CP0_VAL] + len(pychem_sim.conditions) * [INTERCEPT_N] + len(pychem_sim.conditions) * [INTERCEPT_U] + len(pychem_sim.conditions) *[SLOPE_N] + len(pychem_sim.conditions) * [SLOPE_U] + len(pychem_sim.conditions) * [EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]), 'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2) })

    return "No fitting has been performed yet."


### Monomer

In [3]:
pychem_sim = aux_create_pychem_sim_two_state(params,concs, temp_range, model="Monomer", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_global()
pychem_sim.fit_thermal_unfolding_global_global()
pychem_sim.fit_thermal_unfolding_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [4]:
print(parameter_fitting_evaluation_two_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                            Tm (°C)   70.0000   70.035498   
1                                      ΔH (kcal/mol)  250.0000  246.175330   
2                                   Cp (kcal/mol/°C)    1.0000    0.100000   
3                intercept_native - Simulated Signal   24.0000   24.000127   
4              intercept_unfolded - Simulated Signal   -4.0000    0.000187   
5               slope_term_native - Simulated Signal   -0.2700   -0.270038   
6  pre_exponential_factor_unfolded - Simulated Si...   80.5000   87.341028   
7  exponential_coefficient_unfolded - Simulated S...    0.0224    0.027305   

   Relative Difference  
0             0.000507  
1             0.015417  
2             1.636364  
3             0.000005  
4            -2.000187  
5            -0.000141  
6             0.081518  
7             0.197378  


### Dimer

In [5]:
pychem_sim = aux_create_pychem_sim_two_state(params,concs, temp_range, model="Dimer", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_global()
pychem_sim.fit_thermal_unfolding_global_global()
pychem_sim.fit_thermal_unfolding_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [6]:
print(parameter_fitting_evaluation_two_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                            Tm (°C)   70.0000   70.005268   
1                                      ΔH (kcal/mol)  250.0000  249.363656   
2                                   Cp (kcal/mol/°C)    1.0000    0.919342   
3                intercept_native - Simulated Signal   24.0000   23.999059   
4              intercept_unfolded - Simulated Signal   -4.0000   -4.235080   
5               slope_term_native - Simulated Signal   -0.2700   -0.269925   
6  pre_exponential_factor_unfolded - Simulated Si...   80.5000   80.370385   
7  exponential_coefficient_unfolded - Simulated S...    0.0224    0.022191   

   Relative Difference  
0             0.000075  
1             0.002549  
2             0.084048  
3             0.000039  
4            -0.057092  
5            -0.000277  
6             0.001611  
7             0.009377  


#### Experimental data

In [7]:
# creating a Sample object for the storage and processing of the DSF data
sample = ThermalOligomer()

#Setting Oligomeric Model
sample.set_model("Dimer")

# reading the data
sample.read_file('../data/arc_temp_pychemelt_adjusted.csv')

#Adjusting the data to be processable by PyCheMelt
for key in sample.temp_dic.keys():
    for i in range(len(sample.signal_dic[key])):
        mask = ~np.isnan(sample.signal_dic[key][i])

        sample.temp_dic[key][i] = sample.temp_dic[key][i][mask]
        sample.signal_dic[key][i] = sample.signal_dic[key][i][mask]

# Adjusting the scale of the concentration
#sample.set_concentrations([4*1e-6, 20*1e-6])

sample.set_concentrations([4, 20])

# Selecting conditions
sample.set_signal(['Fluorescence'])
sample.select_conditions()
sample.expand_multiple_signal()



# As the data consists of fraction unfolded data we use constant baselines

sample.estimate_baseline_parameters(
    native_baseline_type='constant',
    unfolded_baseline_type='constant',
)

# estimations of parameters
sample.estimate_derivative()
sample.guess_Tm()

# Setting the number of residues of the protein for an initial estimate of the Cp value
sample.n_residues = 53 * 2
sample.guess_Cp()

sample.set_signal_id()

sample.fit_thermal_unfolding_global()
sample.fit_thermal_unfolding_global_global()
#sample.fit_thermal_unfolding_global_global_global(model_scale_factor=True) # Does not show correct fitting

plot_unfolding(sample)

In [8]:
print(sample.params_df)

                                Parameter         Value  Relative error (%)  \
0                                 Tm (°C)  4.417828e+01        5.788693e-01   
1                           ΔH (kcal/mol)  6.473272e+01        4.312871e+00   
2                        Cp (kcal/mol/°C)  1.000000e-01        2.781465e+02   
3     intercept_native - 4 - Fluorescence  5.092665e-03        3.052652e+01   
4    intercept_native - 20 - Fluorescence  3.827767e-20        5.421048e+17   
5   intercept_unfolded - 4 - Fluorescence  1.268930e-01        4.442181e-01   
6  intercept_unfolded - 20 - Fluorescence  2.576480e-02        7.193485e-01   

   Fitting low Bound  Fitting high Bound  
0          32.178276           84.426184  
1          10.000000          500.000000  
2           0.100000            5.000000  
3           0.000000           10.000000  
4           0.000000           10.000000  
5        -100.000000          100.000000  
6         -10.000000           10.000000  


### Trimer

In [9]:
pychem_sim = aux_create_pychem_sim_two_state(params,concs, temp_range, model="Trimer", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_global()
pychem_sim.fit_thermal_unfolding_global_global()
pychem_sim.fit_thermal_unfolding_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [10]:
print(parameter_fitting_evaluation_two_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                            Tm (°C)   70.0000   69.990686   
1                                      ΔH (kcal/mol)  250.0000  250.321350   
2                                   Cp (kcal/mol/°C)    1.0000    1.021641   
3                intercept_native - Simulated Signal   24.0000   23.999070   
4              intercept_unfolded - Simulated Signal   -4.0000   -3.983011   
5               slope_term_native - Simulated Signal   -0.2700   -0.270119   
6  pre_exponential_factor_unfolded - Simulated Si...   80.5000   80.526573   
7  exponential_coefficient_unfolded - Simulated S...    0.0224    0.022422   

   Relative Difference  
0             0.000133  
1             0.001285  
2             0.021409  
3             0.000039  
4            -0.004256  
5            -0.000440  
6             0.000330  
7             0.000962  


### Tetramer

In [11]:
pychem_sim = aux_create_pychem_sim_two_state(params,concs, temp_range, model="Tetramer", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_global()
pychem_sim.fit_thermal_unfolding_global_global()
pychem_sim.fit_thermal_unfolding_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [12]:
print(parameter_fitting_evaluation_two_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                            Tm (°C)   70.0000   69.949116   
1                                      ΔH (kcal/mol)  250.0000  250.946582   
2                                   Cp (kcal/mol/°C)    1.0000    1.041851   
3                intercept_native - Simulated Signal   24.0000   23.991123   
4              intercept_unfolded - Simulated Signal   -4.0000   -4.023543   
5               slope_term_native - Simulated Signal   -0.2700   -0.271470   
6  pre_exponential_factor_unfolded - Simulated Si...   80.5000   80.509215   
7  exponential_coefficient_unfolded - Simulated S...    0.0224    0.022385   

   Relative Difference  
0             0.000727  
1             0.003779  
2             0.040993  
3             0.000370  
4            -0.005868  
5            -0.005428  
6             0.000114  
7             0.000656  


## Three State

In [13]:
def aux_create_pychem_sim_three_state(params, concs, temp_range, model, intermediate, n_residues, rng_seed=2):

    """
    Generate a Pychemelt ThermalOligomer object with simulated data with a two state unfolding model.

    Parameters
    ---------
    params : dict
        dictionary containing the parameters for the simulated signal (e.g., Tm, DH)
    concs : list
        List of concentrations to simulate
    temp_range : list
        Range of temperatures to simulate
    model : str
        Which protein model to use (choices rang from Monomer to Tetramer)
    n_residues : int
        How many residues the simulated protein has.

    Returns
    ------
        pychem.ThermalOligomer()
    """

    rng = np.random.default_rng(rng_seed)

    temp_range_K = temp_range + 273.15


    signal_list = []
    temp_list   = []

    signal_fx = map_three_state_model_to_signal_fx(model + "_" + intermediate + "_intermediate")

    for i,C in enumerate(concs):

        y = signal_fx(temp_range_K, C, **params)


        # Add gaussian error to simulated signal
        y += rng.normal(0, 0.002*1e-3, len(y))

        # Add error to the initial signal to model variance across positions
        #y *= rng.uniform(0.9,1.1)

        signal_list.append(y)
        temp_list.append(temp_range)

    pychem_sim = ThermalOligomer()

    pychem_sim.signal_dic['Simulated signal'] = signal_list
    pychem_sim.temp_dic['Simulated signal']   = [temp_range for _ in range(len(concs))]

    pychem_sim.set_model(model, intermediate)

    pychem_sim.conditions = concs

    pychem_sim.global_min_temp = np.min(temp_range)
    pychem_sim.global_max_temp = np.max(temp_range)

    pychem_sim.set_concentrations()

    pychem_sim.set_signal('Simulated signal')

    pychem_sim.select_conditions()
    pychem_sim.expand_multiple_signal()

    pychem_sim.estimate_baseline_parameters(
        native_baseline_type='linear',
        unfolded_baseline_type='exponential'
    )


    pychem_sim.n_residues = n_residues # only for cp initial guess
    pychem_sim.guess_Cp()

    return pychem_sim

# Centralized test constants
RNG_SEED = 2
TEMP_START = 20.0
TEMP_STOP = 90.0
N_TEMPS = 150
CONCS = np.arange(10, 60, 10)*1e-6
MAX_POINTS = 400


# Model / ground-truth parameters
DHm_VAL_1 = 300
DHm_VAL_2 = 300
Tm_VAL_1 = 50
Tm_VAL_2 = 70
CP_TH = 1.0
CP_1 = 0.5

INTERCEPT_I = 20

INTERCEPT_N = 24
SLOPE_N = -0.27
C_N_VAL = 0
INTERCEPT_U = -4
SLOPE_U = 80.5
EXPONENT_U = 0.0224
C_U_VAL = 0

params = {
    'DH1': DHm_VAL_1,
    'DH2': DHm_VAL_2,
    'T1': Tm_VAL_1+273.15,
    'T2': Tm_VAL_2+273.15,
    'bI': INTERCEPT_I,
    'p1_N': C_N_VAL,
    'p2_N': INTERCEPT_N,
    'p3_N': SLOPE_N,
    'p4_N': 0,
    'p1_U': C_U_VAL,
    'p2_U': INTERCEPT_U,
    'p3_U': SLOPE_U,
    'p4_U': EXPONENT_U,
    'baseline_N_fx':linear_baseline,
    'baseline_U_fx':exponential_baseline,
    'Cp1': CP_1,
    'CpTh': CP_TH,
}


temp_range  = np.linspace(TEMP_START, TEMP_STOP, N_TEMPS)
concs = CONCS

def parameter_fitting_evaluation_three_state(pychem_sim):

    if pychem_sim.global_global_global_fit_done == True:
        actual_params = [Tm_VAL_1, DHm_VAL_1, Tm_VAL_2, DHm_VAL_2, INTERCEPT_N, INTERCEPT_U, INTERCEPT_I, SLOPE_N, SLOPE_U, EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]),'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2)})

    if pychem_sim.global_global_fit_done == True:
        actual_params = [Tm_VAL_1, DHm_VAL_1, Tm_VAL_2, DHm_VAL_2] + len(pychem_sim.conditions) * [INTERCEPT_I] + len(pychem_sim.conditions) * [INTERCEPT_N] + len(pychem_sim.conditions) * [INTERCEPT_U] + [SLOPE_N, SLOPE_U, EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]), 'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2)})


    if pychem_sim.global_fit_done == True:
        actual_params = [Tm_VAL_1, DHm_VAL_1, Tm_VAL_2, DHm_VAL_2] + len(pychem_sim.conditions) * [INTERCEPT_I] + len(pychem_sim.conditions) * [INTERCEPT_N] + len(pychem_sim.conditions) * [INTERCEPT_U] + len(pychem_sim.conditions) *[SLOPE_N] + len(pychem_sim.conditions) * [SLOPE_U] + len(pychem_sim.conditions) * [EXPONENT_U]

        fitted_params = list(pychem_sim.params_df.iloc[:len(actual_params),1])

        return pd.DataFrame(data={'Parameter':list(pychem_sim.params_df.iloc[:len(actual_params),0]), 'Actual': actual_params, 'Fitted': fitted_params, "Relative Difference": abs(np.array(actual_params) - np.array(fitted_params)) / ((np.array(actual_params) + np.array(fitted_params))/2) })

    return "Not fitting has been performed yet."

### Monomer

In [14]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Monomer", intermediate="monomeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [15]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   49.997379   
1                                     ΔH1 (kcal/mol)  300.0000  303.311789   
2                                           Tm2 (°C)   70.0000   70.072726   
3                                     ΔH2 (kcal/mol)  300.0000  288.597816   
4                intercept_native - Simulated signal   24.0000   23.998735   
5              intercept_unfolded - Simulated signal   -4.0000    0.000183   
6          intercept_intermediate - Simulated signal   20.0000   19.997102   
7               slope_term_native - Simulated signal   -0.2700   -0.269887   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   86.907700   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.027221   

   Relative Difference  
0             0.000052  
1             0.010979  
2             0.001038  
3             0.038744  
4             0.

### Dimer

#### Monomeric Intermediate

In [16]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Dimer", intermediate="monomeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [17]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   50.034172   
1                                     ΔH1 (kcal/mol)  300.0000  296.803613   
2                                           Tm2 (°C)   70.0000   70.008963   
3                                     ΔH2 (kcal/mol)  300.0000  298.895160   
4                intercept_native - Simulated signal   24.0000   23.999023   
5              intercept_unfolded - Simulated signal   -4.0000   -4.117138   
6          intercept_intermediate - Simulated signal   20.0000   20.000517   
7               slope_term_native - Simulated signal   -0.2700   -0.269769   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   80.504067   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.022312   

   Relative Difference  
0             0.000683  
1             0.010712  
2             0.000128  
3             0.003690  
4             0.

#### Dimeric Intermediate

In [18]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Dimer", intermediate="dimeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [19]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   50.000323   
1                                     ΔH1 (kcal/mol)  300.0000  302.241215   
2                                           Tm2 (°C)   70.0000   70.057929   
3                                     ΔH2 (kcal/mol)  300.0000  295.742130   
4                intercept_native - Simulated signal   24.0000   23.998859   
5              intercept_unfolded - Simulated signal   -4.0000   -4.113184   
6          intercept_intermediate - Simulated signal   20.0000   20.002789   
7               slope_term_native - Simulated signal   -0.2700   -0.269900   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   80.464385   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.022307   

   Relative Difference  
0             0.000006  
1             0.007443  
2             0.000827  
3             0.014294  
4             0.

### Trimer

#### Monomeric Intermediate

In [20]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Trimer", intermediate="monomeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [21]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   50.117822   
1                                     ΔH1 (kcal/mol)  300.0000  294.241291   
2                                           Tm2 (°C)   70.0000   70.005987   
3                                     ΔH2 (kcal/mol)  300.0000  299.282488   
4                intercept_native - Simulated signal   24.0000   24.004366   
5              intercept_unfolded - Simulated signal   -4.0000   -4.079603   
6          intercept_intermediate - Simulated signal   20.0000   20.000440   
7               slope_term_native - Simulated signal   -0.2700   -0.267014   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   80.500812   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.022340   

   Relative Difference  
0             0.002354  
1             0.019382  
2             0.000086  
3             0.002395  
4             0.

#### Trimeric Intermediate

In [22]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Trimer", intermediate="trimeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [23]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   49.981863   
1                                     ΔH1 (kcal/mol)  300.0000  301.694029   
2                                           Tm2 (°C)   70.0000   70.154140   
3                                     ΔH2 (kcal/mol)  300.0000  293.431185   
4                intercept_native - Simulated signal   24.0000   23.998217   
5              intercept_unfolded - Simulated signal   -4.0000   -3.900858   
6          intercept_intermediate - Simulated signal   20.0000   20.080479   
7               slope_term_native - Simulated signal   -0.2700   -0.269722   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   80.569859   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.022496   

   Relative Difference  
0             0.000363  
1             0.005631  
2             0.002200  
3             0.022138  
4             0.

### Tetramer

In [24]:
pychem_sim = aux_create_pychem_sim_three_state(params,concs, temp_range, model="Tetramer", intermediate="monomeric", n_residues=80, rng_seed=RNG_SEED)

pychem_sim.fit_thermal_unfolding_three_state_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global()
pychem_sim.fit_thermal_unfolding_three_state_global_global_global(model_scale_factor=True)

plot_unfolding(pychem_sim)

In [25]:
print(parameter_fitting_evaluation_three_state(pychem_sim))

                                           Parameter    Actual      Fitted  \
0                                           Tm1 (°C)   50.0000   49.884482   
1                                     ΔH1 (kcal/mol)  300.0000  297.932422   
2                                           Tm2 (°C)   70.0000   70.146453   
3                                     ΔH2 (kcal/mol)  300.0000  247.246429   
4                intercept_native - Simulated signal   24.0000   24.239387   
5              intercept_unfolded - Simulated signal   -4.0000    0.618041   
6          intercept_intermediate - Simulated signal   20.0000   19.986034   
7               slope_term_native - Simulated signal   -0.2700   -0.213261   
8  pre_exponential_factor_unfolded - Simulated si...   80.5000   89.964171   
9  exponential_coefficient_unfolded - Simulated s...    0.0224    0.028418   

   Relative Difference  
0             0.002313  
1             0.006916  
2             0.002090  
3             0.192796  
4             0.

The models were able to fit the simulated data adequately and produced thermodynamic parameters close to the real ones with the exception of the Cp value. The Cp value cannot be very accurately fitted with this method and should preferably by given by the user for an additional benefit in fitting.

Additionally the test on the real world data showed that it can also fit to limited data not in a typical format preferred by this method, as it was written for the original raw data of a DSF machine and here we had Fraction Unfolded.

The fitting to Simulated data demonstrates the possibility of this method to be able to fit to DSF curves following the theoretical basis of the formula.